<a href="https://colab.research.google.com/github/doctoradocarlitoscarrancoq/Data_MPF_DF2/blob/main/MPD_DF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne neurokit2 seaborn scipy umap-learn jinja2 plotly -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import neurokit2 as nk

import seaborn as sns
import matplotlib.pyplot as plt

from glob import glob
from scipy.spatial.distance import pdist
from scipy.spatial.distance import squareform

from scipy.stats import wasserstein_distance

from sklearn.preprocessing import StandardScaler
import umap

from jinja2 import Template

In [ ]:
EEG_PATH="/content/drive/MyDrive/DATASET/MPD_DF/EEG"
PSG_PATH="/content/drive/MyDrive/DATASET/MPD_DF/PSG"

In [ ]:
eeg_files=sorted(
    glob(os.path.join(EEG_PATH,"*.edf"))
)

psg_files=sorted(
    glob(os.path.join(PSG_PATH,"*.edf"))
)

print("EEG:",len(eeg_files))
print("PSG:",len(psg_files))

In [ ]:
def get_subject_id(filepath):

    name=os.path.basename(filepath)

    return name.split("_")[2]

In [ ]:
summary=[]

for file in eeg_files:

    raw=mne.io.read_raw_edf(
        file,
        preload=False,
        verbose=False
    )

    summary.append({

        "Subject":get_subject_id(file),

        "Channels":len(raw.ch_names),

        "SamplingRate":raw.info["sfreq"],

        "DurationMinutes":
            raw.times[-1]/60
    })

dataset_info=pd.DataFrame(summary)

dataset_info

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    data=dataset_info,
    x="Subject",
    y="DurationMinutes"
)

plt.title(
    "Duración EEG por sujeto"
)

plt.show()

In [ ]:
common_channels=None

for file in eeg_files:

    raw=mne.io.read_raw_edf(
        file,
        preload=False,
        verbose=False
    )

    ch=set(raw.ch_names)

    if common_channels is None:
        common_channels=ch
    else:
        common_channels &= ch

common_channels=sorted(
    list(common_channels)
)

print(common_channels)

In [ ]:
all_band_features=[]

bands={

    "Delta":(1,4),
    "Theta":(4,8),
    "Alpha":(8,13),
    "Beta":(13,30)
}

In [ ]:
for file in eeg_files:

    raw=mne.io.read_raw_edf(
        file,
        preload=True,
        verbose=False
    )

    raw.pick(common_channels)

    raw.filter(
        1,
        40,
        verbose=False
    )

    row={
        "Subject":
        get_subject_id(file)
    }

    for band,(fmin,fmax) in bands.items():

        psd=raw.compute_psd(
            fmin=fmin,
            fmax=fmax
        )

        row[band]=psd.get_data().mean()

    all_band_features.append(row)

band_df=pd.DataFrame(
    all_band_features
)

band_df

In [ ]:
band_df["ThetaBeta"]=(
    band_df["Theta"] /
    band_df["Beta"]
)

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    data=band_df,
    x="Subject",
    y="ThetaBeta"
)

plt.title(
    "Theta/Beta por sujeto"
)

plt.show()

#Heatmap inter-sujeto

In [ ]:
matrix=squareform(

    pdist(
        band_df[
            ["Delta",
             "Theta",
             "Alpha",
             "Beta"]
        ],
        metric="euclidean"
    )
)

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    matrix,
    annot=True,
    cmap="viridis"
)

plt.title(
    "Distancia Inter-Sujeto"
)

plt.show()

#UMAP

In [ ]:
reducer=umap.UMAP(
    random_state=42
)

In [ ]:
embedding=reducer.fit_transform(

    band_df[
        ["Delta",
         "Theta",
         "Alpha",
         "Beta"]
    ]
)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    embedding[:,0],
    embedding[:,1]
)

for i,sub in enumerate(
    band_df["Subject"]
):

    plt.text(
        embedding[i,0],
        embedding[i,1],
        sub
    )

plt.title(
    "UMAP Sujetos MPD-DF"
)

plt.show()

#ECG

In [ ]:
raw_psg=mne.io.read_raw_edf(
    psg_files[0],
    preload=True
)

In [ ]:
ecg_idx=raw_psg.ch_names.index(
    "ECG"
)

In [ ]:
ecg=raw_psg.get_data()[ecg_idx]

signals,info=nk.ecg_process(
    ecg,
    sampling_rate=1024
)

In [ ]:
nk.ecg_plot(signals)

#HRV

In [ ]:
hrv=nk.hrv(info)

hrv.T

#EOG

In [ ]:
eog1=raw_psg.get_data()[0]

eog2=raw_psg.get_data()[1]

In [ ]:
plt.figure(figsize=(15,4))

plt.plot(
    eog1[:20000],
    label="E1-M2"
)

plt.plot(
    eog2[:20000],
    label="E2-M2"
)

plt.legend()

plt.title(
    "Actividad Ocular"
)

plt.show()

#Respiración

In [ ]:
thor_idx=raw_psg.ch_names.index(
    "Thor"
)

thor=raw_psg.get_data()[thor_idx]

In [ ]:
plt.figure(figsize=(15,4))

plt.plot(
    thor[:20000]
)

plt.title(
    "Respiración Torácica"
)

plt.show()

#Topomaps

Como MPD-DF no contiene posiciones 3D, utilice:

In [ ]:
montage = mne.channels.make_standard_montage(
    "standard_1020"
)

raw.set_montage(
    montage,
    on_missing="ignore"
)

#Topomap promedio Theta

In [ ]:
theta_maps=[]

In [ ]:
for file in eeg_files:

    raw=mne.io.read_raw_edf(
        file,
        preload=True,
        verbose=False
    )

    raw.set_montage(
        montage,
        on_missing="ignore"
    )

    psd=raw.compute_psd(
        fmin=4,
        fmax=8
    )

    theta_maps.append(

        psd.get_data().mean(
            axis=1
        )
    )

In [ ]:
theta_mean=np.mean(
    theta_maps,
    axis=0
)

In [ ]:
fig,ax=plt.subplots(
    figsize=(7,7)
)

mne.viz.plot_topomap(
    theta_mean,
    raw.info,
    axes=ax,
    show=False
)

plt.title(
    "Theta Promedio MPD-DF"
)

plt.show()

#Variabilidad inter-sujeto

In [ ]:
theta_std=np.std(
    theta_maps,
    axis=0
)

In [ ]:
fig,ax=plt.subplots(
    figsize=(7,7)
)

mne.viz.plot_topomap(
    theta_std,
    raw.info,
    axes=ax,
    cmap="Reds",
    show=False
)

plt.title(
    "Variabilidad Theta"
)

plt.show()

#Exportar HTML

In [ ]:
dataset_info.to_html(
    "dataset_summary.html"
)

In [ ]:
for f in eeg_files:

    raw = mne.io.read_raw_edf(
        f,
        preload=False,
        verbose=False
    )

    print("="*50)
    print(os.path.basename(f))
    print("Canales:", len(raw.ch_names))
    print(raw.ch_names)

In [ ]:
raw = mne.io.read_raw_edf(
    eeg_files[0],
    preload=False,
    verbose=False
)

print(raw.ch_names)

In [ ]:
raw.set_montage(
    mne.channels.make_standard_montage(
        "standard_1020"
    ),
    on_missing="ignore"
)

In [ ]:
raw = mne.io.read_raw_edf(
    file,
    preload=True,
    verbose=False
)

montage = mne.channels.make_standard_montage(
    "standard_1020"
)

raw.set_montage(
    montage,
    on_missing="warn"
)

In [ ]:
print(raw.get_montage())

In [ ]:
picks = mne.pick_types(
    raw.info,
    eeg=True
)

pos = np.array([
    raw.info["chs"][k]["loc"][:2]
    for k in picks
])

pos.shape

In [ ]:
import mne
import numpy as np

file = eeg_files[0]

raw = mne.io.read_raw_edf(
    file,
    preload=True,
    verbose=False
)

montage = mne.channels.make_standard_montage(
    "standard_1020"
)

raw.set_montage(
    montage,
    on_missing="ignore"
)

psd_theta = raw.compute_psd(
    fmin=4,
    fmax=8
)

theta_power = np.mean(
    psd_theta.get_data(),
    axis=1
)

print(theta_power.shape)

In [ ]:
mne.viz.plot_topomap(
    theta_power,
    pos
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(6,6)
)

mne.viz.plot_topomap(
    theta_power,
    pos,
    axes=ax,
    contours=8,
    cmap="viridis",
    show=False
)

plt.title(
    "Theta Power (4-8 Hz)"
)

plt.show()

In [ ]:
def create_band_topomap(
    eeg_file,
    fmin,
    fmax,
    title
):

    raw = mne.io.read_raw_edf(
        eeg_file,
        preload=True,
        verbose=False
    )

    raw.set_montage(
        mne.channels.make_standard_montage(
            "standard_1020"
        ),
        on_missing="ignore"
    )

    psd = raw.compute_psd(
        fmin=fmin,
        fmax=fmax
    )

    power = psd.get_data().mean(
        axis=1
    )

    picks = mne.pick_types(
        raw.info,
        eeg=True
    )

    pos = np.array([
        raw.info["chs"][k]["loc"][:2]
        for k in picks
    ])

    fig, ax = plt.subplots(
        figsize=(6,6)
    )

    mne.viz.plot_topomap(
        power,
        pos,
        axes=ax,
        show=False
    )

    plt.title(title)

    plt.show()

    return power

In [ ]:
theta_power = create_band_topomap(
    eeg_files[0],
    4,
    8,
    "Theta"
)

In [ ]:
alpha_power = create_band_topomap(
    eeg_files[0],
    8,
    13,
    "Alpha"
)


In [ ]:
beta_power = create_band_topomap(
    eeg_files[0],
    13,
    30,
    "Beta"
)
